# Geospatial Site Scoring — Tech Talk Demo

**ML-powered revenue prediction and lookalike classification for 60K+ gas station advertising sites.**

This notebook walks through the end-to-end pipeline:
1. Data loading & exploration
2. Feature engineering
3. Model training (XGBoost & Neural Network)
4. Evaluation & interpretation

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Project paths
PROJECT_ROOT = Path('.').resolve()
DATA_INPUT = PROJECT_ROOT / 'data' / 'input'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

print(f'Project root: {PROJECT_ROOT}')
print(f'Input files: {[f.name for f in DATA_INPUT.iterdir() if f.is_file()]}')

## 1. The Data Landscape

| File | Size | Description |
|------|------|-------------|
| `site_scores_revenue_and_diagnostics.csv` | 927 MB | Monthly records: 1.4M rows × 94 cols |
| `nearest_site_distances.csv` | 6.5 MB | Distance to nearest competing site |
| `site_interstate_distances.csv` | 3.8 MB | Distance to nearest interstate |
| `site_kroger_distances.csv` | 2.7 MB | Distance to nearest Kroger |
| `site_mcdonalds_distances.csv` | 2.8 MB | Distance to nearest McDonald's |
| `site_walmart_distances.csv` | 2.8 MB | Distance to nearest Walmart |
| `site_target_distances.csv` | 2.8 MB | Distance to nearest Target |

In [ ]:
# Load the pre-processed training data (already transformed by data_transform.py)
train_df = pl.read_parquet(DATA_PROCESSED / 'site_training_data.parquet')
print(f'Training data: {train_df.shape[0]:,} sites × {train_df.shape[1]} columns')
print(f'\nColumn types:')
print(f'  Numeric:     {len([c for c in train_df.columns if train_df[c].dtype in [pl.Float64, pl.Float32]])}')
print(f'  Integer:     {len([c for c in train_df.columns if train_df[c].dtype in [pl.Int8, pl.Int16, pl.Int32, pl.Int64]])}')
print(f'  String:      {len([c for c in train_df.columns if train_df[c].dtype == pl.Utf8])}')
print(f'  Log-transformed: {len([c for c in train_df.columns if c.startswith("log_")])}')
print(f'  Encoded flags:   {len([c for c in train_df.columns if c.endswith("_encoded")])}')
print(f'  RS momentum:     {len([c for c in train_df.columns if c.startswith("rs_")])}')

## 2. Revenue Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

revenue = train_df['avg_monthly_revenue'].to_numpy()

# Raw distribution
axes[0].hist(revenue, bins=100, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].set_title('Avg Monthly Revenue (Raw)')
axes[0].set_xlabel('Revenue ($)')
axes[0].axvline(np.median(revenue), color='red', linestyle='--', label=f'Median: ${np.median(revenue):,.0f}')
axes[0].legend()

# Log distribution
log_rev = np.log1p(revenue[revenue > 0])
axes[1].hist(log_rev, bins=80, color='coral', edgecolor='none', alpha=0.8)
axes[1].set_title('Log(Revenue) — More Normal')
axes[1].set_xlabel('log(1 + revenue)')

# Percentile view
pcts = np.percentile(revenue, np.arange(0, 101, 5))
axes[2].bar(range(len(pcts)), pcts, color='seagreen', alpha=0.8)
axes[2].set_title('Revenue by Percentile')
axes[2].set_xlabel('Percentile')
axes[2].set_ylabel('Revenue ($)')
axes[2].set_xticks(range(0, len(pcts), 4))
axes[2].set_xticklabels([f'p{i}' for i in range(0, 101, 20)])

plt.tight_layout()
plt.show()

print(f'Revenue stats: min=${revenue.min():,.0f}, median=${np.median(revenue):,.0f}, '
      f'mean=${revenue.mean():,.0f}, p90=${np.percentile(revenue, 90):,.0f}, max=${revenue.max():,.0f}')

## 3. Feature Engineering Highlights

### Relative Strength (Momentum Indicators)
Compares recent performance to historical average — values > 1.0 indicate upward trend.

In [ ]:
rs_cols = [c for c in train_df.columns if c.startswith('rs_')]
rs_data = train_df.select(rs_cols).to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# RS distribution
for col in rs_cols:
    axes[0].hist(rs_data[col].dropna(), bins=50, alpha=0.4, label=col.replace('rs_', ''))
axes[0].axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='Neutral (1.0)')
axes[0].set_title('Relative Strength Distributions')
axes[0].set_xlabel('RS Value')
axes[0].legend(fontsize=7)
axes[0].set_xlim(0, 3)

# RS correlation with revenue
corr_data = train_df.select(rs_cols + ['avg_monthly_revenue']).to_pandas().corr()['avg_monthly_revenue'].drop('avg_monthly_revenue')
corr_data.sort_values().plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('RS Feature Correlation with Revenue')
axes[1].set_xlabel('Pearson Correlation')

plt.tight_layout()
plt.show()

### Geospatial Distance Features
Log-transformed distances to key landmarks (interstate, Kroger, McDonald's, Walmart, Target).

In [ ]:
dist_cols = [c for c in train_df.columns if c.startswith('log_min_distance')]
dist_data = train_df.select(dist_cols + ['avg_monthly_revenue']).to_pandas()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(dist_cols):
    if i < len(axes):
        label = col.replace('log_min_distance_to_', '').replace('_mi', '')
        axes[i].scatter(dist_data[col], dist_data['avg_monthly_revenue'],
                       alpha=0.05, s=1, color='steelblue')
        axes[i].set_title(f'Distance to {label.title()}')
        axes[i].set_xlabel(f'log(distance in mi)')
        axes[i].set_ylabel('Avg Monthly Revenue')

# Hide extra subplot
if len(dist_cols) < len(axes):
    axes[-1].set_visible(False)

plt.tight_layout()
plt.show()

## 4. Model Training

The project supports two model types:
- **XGBoost**: Gradient boosted trees (fast, interpretable)
- **Neural Network**: PyTorch with embeddings for categoricals (higher capacity)

And two task types:
- **Regression**: Predict `avg_monthly_revenue` directly
- **Lookalike**: Classify sites as "top performers" (p90+) vs others

In [ ]:
# Quick XGBoost demo on the training data
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from site_scoring.config import Config
from site_scoring.data_loader import DataProcessor

config = Config()
print(f'Task: {config.task_type}')
print(f'Target: {config.target}')
print(f'Device: {config.device}')
print(f'Features: {len(config.numeric_features)} numeric + {len(config.categorical_features)} categorical + {len(config.boolean_features)} boolean')

## 5. Key Architectural Decisions

| Decision | Choice | Why |
|----------|--------|-----|
| CSV parsing | **Polars** (not pandas) | 10-20x faster for 927MB file |
| Aggregation | One row per site | Prevents data leakage from monthly records |
| Revenue target | `avg_monthly_revenue` | Normalizes for site age |
| Log transforms | Sign-preserving `log(1+\|x\|)` | Handles skewed distributions |
| Flag encoding | One-hot (Yes/No → 1/0) | 40 boolean features from restrictions/capabilities |
| High cardinality | Top-N binning | Retailer (30), brand_c_store (30), brand_fuel (10) |
| Momentum | Multi-horizon RS | 3/6mo, 6/12mo, 12/24mo windows |

In [ ]:
# Summary statistics for the training dataset
print('=== Training Dataset Summary ===')
print(f'Sites: {train_df.shape[0]:,}')
print(f'Columns: {train_df.shape[1]}')
print(f'\nActive months: median={train_df["active_months"].median():.0f}, '
      f'mean={train_df["active_months"].mean():.1f}')
print(f'Avg monthly revenue: median=${train_df["avg_monthly_revenue"].median():,.0f}, '
      f'mean=${train_df["avg_monthly_revenue"].mean():,.0f}')

# Network distribution
print(f'\nNetwork distribution:')
for row in train_df.group_by('network').len().sort('len', descending=True).head(5).iter_rows(named=True):
    print(f'  {row["network"]}: {row["len"]:,} sites')